In [1]:
import sqlite3, pandas as pd

conn = sqlite3.connect("../data/crawl.db")
books = pd.read_sql("SELECT * FROM books", conn)
quotes = pd.read_sql("SELECT * FROM quotes", conn)

print("books:", len(books), "| quotes:", len(quotes))
print("books dtypes:\n", books.dtypes)
print("quotes dtypes:\n", quotes.dtypes)

books: 160 | quotes: 190
books dtypes:
 upc                object
title              object
price_excl_tax    float64
availability        int64
reviews             int64
category           object
description        object
rating              int64
product_url        object
dtype: object
quotes dtypes:
 business_key         object
quote                object
author               object
tags                 object
source_tag           object
author_born_date     object
author_born_place    object
author_bio           object
author_url           object
dtype: object


In [2]:
# полнота полей
def completeness(df):
    return (1 - df.isna().mean() - (df == "").mean()) * 100

print("Полнота books (%):\n", completeness(books).round(1))
print("Полнота quotes (%):\n", completeness(quotes).round(1))

Полнота books (%):
 upc               100.0
title             100.0
price_excl_tax    100.0
availability      100.0
reviews           100.0
category          100.0
description       100.0
rating            100.0
product_url       100.0
dtype: float64
Полнота quotes (%):
 business_key         100.0
quote                100.0
author               100.0
tags                 100.0
source_tag           100.0
author_born_date     100.0
author_born_place    100.0
author_bio           100.0
author_url           100.0
dtype: float64


In [3]:
# агрегаты по книгам
books.groupby("category")["price_excl_tax"].agg(["count", "mean", "min", "max"]) \
     .sort_values("mean", ascending=False).head(10)

,count,mean,min,max
category,,,,
christian,1,54.0000,54.00,54.00
childrens,4,51.4350,36.89,58.08
politics,1,51.3300,51.33,51.33
science,2,50.1600,42.96,57.36
health,1,49.0500,49.05,49.05
new adult,3,48.0700,43.29,55.85
self help,2,46.1900,46.03,46.35
travel,1,45.1700,45.17,45.17
art,1,44.1800,44.18,44.18


In [4]:
# агрегаты по цитатам
quotes["tags_list"] = quotes["tags"].fillna("").str.split(",")
exploded = quotes.explode("tags_list")
exploded.groupby("tags_list").size().sort_values(ascending=False).head(10)

tags_list
inspirational    47
life             40
books            40
love             40
humor            32
reading          24
friendship       16
friends          16
writing          14
truth            10
dtype: int64

In [5]:
# топ-авторы
quotes.groupby("author").size().sort_values(ascending=False).head(10)

author
Albert Einstein      17
Mark Twain           13
Jane Austen          13
Marilyn Monroe       11
Dr. Seuss             9
C.S. Lewis            9
John Lennon           8
Elie Wiesel           8
Madeleine L'Engle     7
George Carlin         6
dtype: int64

In [6]:
# распределение рейтингов
books["rating"].value_counts(dropna=False).sort_index()

rating
1    40
2    33
3    29
4    31
5    27
Name: count, dtype: int64